In [12]:
import pandas as pd
df = pd.read_csv('tsla_articles.csv')
df.head()

,Unnamed: 0,date,text
0,0,2021-01-27,The company benefited from a jump in sales of ...
1,1,2021-01-19,"Rivian, which has raised another $2.65 billion..."
2,2,2021-01-15,Traditional automakers have struggled to sell ...
3,3,2021-01-22,The pandemic dampened sales for all automakers...
4,4,2021-01-07,Mr. Musk’s net worth was $188.5 billion at 10:...


In [13]:
from transformers import BertForSequenceClassification, AutoTokenizer, pipeline
# Initialize FinTwitBERT sentiment classifier
def initialize_sentiment_analyzer():
    model = BertForSequenceClassification.from_pretrained(
        "StephanAkkerman/FinTwitBERT-sentiment",
        num_labels=3,
        id2label={0: "neutral", 1: "positive", 2: "negative"},
        label2id={"neutral": 0, "positive": 1, "negative": 2},
    )
    tokenizer = AutoTokenizer.from_pretrained("StephanAkkerman/FinTwitBERT-sentiment")
    return pipeline("text-classification", model=model, tokenizer=tokenizer, return_all_scores=True, device=-1)

sentiment_classifier = initialize_sentiment_analyzer()

def classify_net_sentiment(text):
    try:
        scores = sentiment_classifier(text)[0]
        score_dict = {item['label']: item['score'] for item in scores}
        return score_dict.get("positive", 0) - score_dict.get("negative", 0)
    except:
        return None  # or 0 if you prefer

# Apply to your DataFrame
df["net_sentiment"] = df["text"].apply(classify_net_sentiment)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [14]:
daily_sentiment = df.groupby("date").agg({
    "net_sentiment": "mean"
}).reset_index()

In [15]:
daily_sentiment

,date,net_sentiment
0,2020-01-01,0.899516
1,2020-01-03,0.973886
2,2020-01-04,-0.185437
3,2020-01-09,-0.355640
4,2020-01-10,-0.420654
...,...,...
847,2024-06-18,-0.999800
848,2024-06-21,0.208686
849,2024-06-24,-0.997312
850,2024-06-25,-0.199719


In [16]:
daily_sentiment.to_csv("tsla_sentiment.csv", index=False)

In [17]:
df.shape

(1414, 4)